# QSAR Deep Learning — GNN/CGCNN/LSTM Comparison

Simple GNN (GCN) + CGCNN + SMILES-LSTM vs top 3 classical models.

## 0. Setup

In [ ]:
import pandas as pd, numpy as np, torch, math, os, pickle, warnings
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_add_pool, global_mean_pool
from rdkit import Chem
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score)
from tqdm.notebook import trange
warnings.filterwarnings('ignore')
if torch.cuda.is_available(): device = torch.device('cuda')
elif torch.backends.mps.is_available(): device = torch.device('mps')
else: device = torch.device('cpu')
print(f'Device: {device}')

## 1. Load + binarize

In [ ]:
df = pd.read_csv('data/curated_data_rdkit_Imane.csv')
df.rename(columns={'logC50':'log_LC50','SMILES_curated':'SMILES'}, inplace=True)
meta = df['CAS'].astype(str).str.contains('Meta', na=False)
df = df[~meta].reset_index(drop=True)
df['mol'] = df['SMILES'].apply(lambda s: Chem.MolFromSmiles(s) if s else None)
df = df.dropna(subset=['mol']).reset_index(drop=True)
df['toxic'] = (df['log_LC50'] > 1.0).astype(int)
pos = df['toxic'].sum(); neg = len(df) - pos
print(f'Molecules: {len(df)} | Toxic: {pos} ({pos/len(df)*100:.1f}%) | Non-toxic: {neg} ({neg/len(df)*100:.1f}%)')

## 2. Split + binarize targets

In [ ]:
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=df['toxic'])
y_all = torch.tensor(df['toxic'].values, dtype=torch.float32)
y_train, y_test = y_all[train_idx], y_all[test_idx]
train_idx_list = train_idx.tolist() if isinstance(train_idx, np.ndarray) else train_idx
test_idx_list = test_idx.tolist() if isinstance(test_idx, np.ndarray) else test_idx
smiles_train = df['SMILES'].iloc[train_idx_list].tolist()
smiles_test = df['SMILES'].iloc[test_idx_list].tolist()
print(f'Train: {len(smiles_train)}  Test: {len(smiles_test)}')

## 3. Graph featurization (for GNN + CGCNN)

In [ ]:
ATOM_DIM = 60; MAX_DEG = 6; MAX_HS = 5
HYBRID_MAP = {Chem.HybridizationType.SP:0,Chem.HybridizationType.SP2:1,Chem.HybridizationType.SP3:2,
              Chem.HybridizationType.SP3D:3,Chem.HybridizationType.SP3D2:4}
CHIRAL_MAP = {Chem.ChiralType.CHI_TETRAHEDRAL_CW:0,Chem.ChiralType.CHI_TETRAHEDRAL_CCW:1,Chem.ChiralType.CHI_UNSPECIFIED:2}
BOND_MAP = {Chem.BondType.SINGLE:0,Chem.BondType.DOUBLE:1,Chem.BondType.TRIPLE:2,Chem.BondType.AROMATIC:3}
def atom_feats(a):
    o = [0]*ATOM_DIM; an = a.GetAtomicNum()
    if an < ATOM_DIM: o[an] = 1
    return torch.tensor(o + [min(a.GetDegree(),MAX_DEG)/MAX_DEG, np.clip(a.GetFormalCharge(),-3,3)/3,
                              min(a.GetTotalNumHs(),MAX_HS)/MAX_HS, HYBRID_MAP.get(a.GetHybridization(),0)/4,
                              1.0 if a.GetIsAromatic() else 0.0, CHIRAL_MAP.get(a.GetChiralTag(),0)/2,
                              (a.GetMass()-12)/100], dtype=torch.float)
def bond_feats(b):
    return torch.tensor([BOND_MAP.get(b.GetBondType(),0)/3, 1.0 if b.GetIsConjugated() else 0.0,
                         1.0 if b.IsInRing() else 0.0], dtype=torch.float)  # 3-dim for CGCNN
def mol_to_graph(mol):
    atoms = list(mol.GetAtoms())
    x = torch.stack([atom_feats(a) for a in atoms])
    ei, ea = [], []
    for b in mol.GetBonds():
        i,j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        e = bond_feats(b)
        ei.extend([[i,j],[j,i]]); ea.extend([e,e])
    ei = torch.tensor(ei, dtype=torch.long).T if ei else torch.empty((2,0), dtype=torch.long)
    ea = torch.stack(ea) if ea else torch.empty((0,3), dtype=torch.float)
    return Data(x=x, edge_index=ei, edge_attr=ea)
graphs = [mol_to_graph(m) for m in df['mol']]
g0 = graphs[0]
print(f'Graphs: {len(graphs)}, Node dim: {g0.x.shape[1]}, Edge dim: {g0.edge_attr.shape[1]}')
train_graphs = [graphs[i] for i in train_idx_list]
test_graphs = [graphs[i] for i in test_idx_list]

## 4. Graph DataLoader

In [ ]:
class GraphDataset(Dataset):
    def __init__(self, gs, y): self.gs = gs; self.y = y
    def __len__(self): return len(self.gs)
    def __getitem__(self, i): return self.gs[i], self.y[i]
def collate_graphs(b):
    gs, ts = zip(*b); return Batch.from_data_list(list(gs)), torch.stack(ts)
train_loader = DataLoader(GraphDataset(train_graphs, y_train), 32, shuffle=True, collate_fn=collate_graphs)
test_loader  = DataLoader(GraphDataset(test_graphs, y_test), 32, shuffle=False, collate_fn=collate_graphs)
print(f'Batches: train={len(train_loader)} test={len(test_loader)}')

## 5. Simple GNN (GCN-based)

In [ ]:
class SimpleGNN(nn.Module):
    def __init__(self, nd, hd=128, nl=3, dp=0.2):
        super().__init__()
        self.np = nn.Linear(nd, hd)
        self.convs = nn.ModuleList([GCNConv(hd, hd) for _ in range(nl)])
        self.norms = nn.ModuleList([nn.BatchNorm1d(hd) for _ in range(nl)])
        self.fc = nn.Sequential(nn.Linear(hd, 64), nn.GELU(), nn.Dropout(dp),
                                 nn.Linear(64, 32), nn.GELU(), nn.Dropout(dp), nn.Linear(32, 1))
        self.dp = nn.Dropout(dp)
    def forward(self, d):
        x, ei, b = d.x, d.edge_index, d.batch
        x = self.np(x)
        for c, no in zip(self.convs, self.norms):
            r = x; x = c(x, ei); x = no(x + r); x = F.gelu(x); x = self.dp(x)
        x = global_mean_pool(x, b)
        return self.fc(x).squeeze(-1)

## 6. CGCNN (Crystal Graph CNN)

In [ ]:
class CGCNNConv(nn.Module):
    def __init__(self, nd, ed):
        super().__init__()
        self.lin_z = nn.Linear(2 * nd + ed, nd)
        self.lin_g = nn.Linear(2 * nd + ed, nd)
        self.lin_x = nn.Linear(nd, nd)
    def forward(self, x, ei, ea):
        i, j = ei[0], ei[1]
        z_in = torch.cat([x[i], x[j], ea], dim=-1)
        z = self.lin_z(z_in)
        g = torch.sigmoid(self.lin_g(z_in))
        msg = g * self.lin_x(z)
        out = global_add_pool(msg, i, x.shape[0])
        return F.gelu(x + out)

class CGCNN(nn.Module):
    def __init__(self, nd, ed=3, hd=128, nl=4, dp=0.2):
        super().__init__()
        self.np = nn.Linear(nd, hd)
        
        self.convs = nn.ModuleList([CGCNNConv(hd, ed) for _ in range(nl)])
        self.fc = nn.Sequential(nn.Linear(hd, 64), nn.GELU(), nn.Dropout(dp),
                                 nn.Linear(64, 32), nn.GELU(), nn.Dropout(dp), nn.Linear(32, 1))
        self.dp = nn.Dropout(dp)
    def forward(self, d):
        x, ei, ea, b = d.x, d.edge_index, d.edge_attr, d.batch
        x = self.np(x)
        for c in self.convs: x = c(x, ei, ea); x = self.dp(x)
        x = global_mean_pool(x, b)
        return self.fc(x).squeeze(-1)

## 7. SMILES-LSTM

In [ ]:
# Simple character-level SMILES tokenizer
SMILES_CHARS = 'CAM NnPpSsiI1234567890BDFGcbrfl-+=#()[]/\@.%'
char2idx = {c: i+2 for i, c in enumerate(SMILES_CHARS)}  # 0=pad, 1=unk
char2idx['<PAD>'] = 0; char2idx['<UNK>'] = 1
def tokenize(smiles, max_len=256):
    ids = []
    for s in smiles:
        toks = []
        i = 0
        while i < len(s):
            if i+1 < len(s) and s[i:i+2] in ('Cl','Br','Si','Se','As'): toks.append(s[i:i+2]); i+=2
            elif s[i] in char2idx: toks.append(s[i]); i+=1
            else: toks.append('<UNK>'); i+=1
        ids.append([char2idx.get(t, 1) for t in toks[:max_len]] + [0]*(max_len - len(toks)))
    return torch.tensor(ids, dtype=torch.long)

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size=len(char2idx), emb_dim=128, hd=256, nl=3, dp=0.3):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hd, num_layers=nl, batch_first=True, bidirectional=True, dropout=dp)
        self.fc = nn.Sequential(nn.Linear(hd*2, 128), nn.GELU(), nn.Dropout(dp),
                                 nn.Linear(128, 64), nn.GELU(), nn.Dropout(dp), nn.Linear(64, 1))
    def forward(self, x):
        x = self.emb(x)
        x, (h, c) = self.lstm(x)
        x = torch.cat([h[-2], h[-1]], dim=1)  # concat last forward + backward
        return self.fc(x).squeeze(-1)

class LSTMDataset(Dataset):
    def __init__(self, sm, y, max_len=256):
        self.x = tokenize(sm, max_len); self.y = y
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]
train_lstm = DataLoader(LSTMDataset(smiles_train, y_train), 32, shuffle=True)
test_lstm  = DataLoader(LSTMDataset(smiles_test, y_test), 32, shuffle=False)

## 8. Shared training loop

In [ ]:
def train_model(model, train_loader, test_loader, epochs=200, lr=1e-3, patience=30, name='Model', is_lstm=False):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    total_steps = epochs * len(train_loader); warmup = int(0.1 * total_steps)
    def lr_fn(s): return s/max(1,warmup) if s<warmup else 0.5*(1+math.cos(math.pi*(s-warmup)/max(1,total_steps-warmup)))
    sch = torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)
    crit = nn.BCEWithLogitsLoss()
    best_loss = float('inf'); best_state = None; pc = 0; gs = 0
    pbar = trange(epochs, desc=name, leave=True)
    for ep in pbar:
        model.train(); tl = 0; n = 0
        for b in train_loader:
            if is_lstm: x, y = b[0].to(device), b[1].to(device).unsqueeze(-1)
            else: x, y = b[0].to(device), b[1].to(device).unsqueeze(-1)
            opt.zero_grad(); loss = crit(model(x).unsqueeze(-1), y)
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step(); sch.step(gs); gs += 1
            tl += loss.item() * len(y); n += len(y)
        model.eval(); vl = 0; nv = 0
        with torch.no_grad():
            for b in test_loader:
                if is_lstm: x, y = b[0].to(device), b[1].to(device).unsqueeze(-1)
                else: x, y = b[0].to(device), b[1].to(device).unsqueeze(-1)
                vl += crit(model(x).unsqueeze(-1), y).item() * len(y); nv += len(y)
        pbar.set_postfix({'train':f'{tl/n:.4f}','val':f'{vl/nv:.4f}'})
        if vl < best_loss: best_loss=vl; best_state={k:v.clone() for k,v in model.state_dict().items()}; pc=0
        else: pc+=1
        if pc >= patience: pbar.close(); print(f'  Early stop @ {ep+1}'); break
    if best_state: model.load_state_dict(best_state)
    return model

## 9. Evaluation

In [ ]:
def evaluate(model, loader, is_lstm=False):
    preds, targets = [], []
    model.eval()
    with torch.no_grad():
        for b in loader:
            if is_lstm: x, y = b[0].to(device), b[1].unsqueeze(-1)
            else: x, y = b[0].to(device), b[1]
            logits = model(x)
            logits_ = logits.unsqueeze(-1) if logits.dim() == 1 else logits
            preds.extend(torch.sigmoid(logits_).squeeze(-1).cpu().numpy())
            targets.extend(y.cpu().numpy())
    preds = np.array(preds); targets = np.array(targets)
    binary = (preds > 0.5).astype(int)
    return {'accuracy': accuracy_score(targets, binary),
            'auc_roc': roc_auc_score(targets, preds),
            'sensitivity': recall_score(targets, binary, zero_division=0),
            'specificity': recall_score(1-targets, 1-binary, zero_division=0),
            'precision': precision_score(targets, binary, zero_division=0),
            'f1': f1_score(targets, binary, zero_division=0)}

## 10. Train Simple GNN

In [ ]:
import os
gnn_path = 'models/SimpleGNN_binary.pth'
if os.path.exists(gnn_path):
    print('=== Simple GNN (GCN) — loading checkpoint ===')
    gnn = SimpleGNN(nd=g0.x.shape[1]).to(device)
    gnn.load_state_dict(torch.load(gnn_path, map_location=device))
else:
    print('=== Simple GNN (GCN) — training ===')
    gnn = SimpleGNN(nd=g0.x.shape[1]).to(device)
    gnn = train_model(gnn, train_loader, test_loader, name='GNN')
    torch.save(gnn.state_dict(), gnn_path)
res_gnn = evaluate(gnn, test_loader)
print('GNN:', {k: f'{v:.4f}' for k,v in res_gnn.items()})

## 11. Train CGCNN

In [ ]:
cgcnn_path = 'models/CGCNN_binary.pth'
if os.path.exists(cgcnn_path):
    print('\n=== CGCNN — loading checkpoint ===')
    cgcnn = CGCNN(nd=g0.x.shape[1]).to(device)
    cgcnn.load_state_dict(torch.load(cgcnn_path, map_location=device))
else:
    print('\n=== CGCNN — training ===')
    cgcnn = CGCNN(nd=g0.x.shape[1]).to(device)
    cgcnn = train_model(cgcnn, train_loader, test_loader, name='CGCNN')
    torch.save(cgcnn.state_dict(), cgcnn_path)
res_cgcnn = evaluate(cgcnn, test_loader)
print('CGCNN:', {k: f'{v:.4f}' for k,v in res_cgcnn.items()})

## 12. Train SMILES-LSTM

In [ ]:
lstm_path = 'models/LSTM_binary.pth'
if os.path.exists(lstm_path):
    print('\n=== SMILES-LSTM — loading checkpoint ===')
    lstm = LSTMClassifier().to(device)
    lstm.load_state_dict(torch.load(lstm_path, map_location=device))
else:
    print('\n=== SMILES-LSTM — training ===')
    lstm = LSTMClassifier().to(device)
    lstm = train_model(lstm, train_lstm, test_lstm, name='LSTM', is_lstm=True)
    torch.save(lstm.state_dict(), lstm_path)
res_lstm = evaluate(lstm, test_lstm, is_lstm=True)
print('LSTM:', {k: f'{v:.4f}' for k,v in res_lstm.items()})

## 13. Load classical benchmark

In [ ]:
ck = pickle.load(open('models/classical_benchmark.pkl','rb'))
classical_res = {r['model']: r for r in ck['results']}
top3_names = ck['top3']
print(f'Top 3 classical: {top3_names}')
for n in top3_names:
    r = classical_res[n]
    print(f'  {n:25s} AUC={r["auc_roc"]:.4f}  Acc={r["accuracy"]:.4f}  F1={r["f1"]:.4f}')

## 14. Comparison: DL vs Top-3 Classical

In [ ]:
dl_res = {'Simple GNN': res_gnn, 'CGCNN': res_cgcnn, 'LSTM': res_lstm}
combined = {}
for n in top3_names: combined[f'Classical: {n}'] = classical_res[n]
for n, r in dl_res.items(): combined[f'DL: {n}'] = r
dfc = pd.DataFrame(combined).T
metrics = ['auc_roc','accuracy','f1','sensitivity','specificity','precision']
print(dfc[metrics].to_string())
print(f'\nBest AUC-ROC: {dfc["auc_roc"].idxmax()} ({dfc["auc_roc"].max():.4f})')

## 15. Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
x = np.arange(len(dfc)); colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#937860']
for i, (ax, m, t) in enumerate(zip(axes, ['auc_roc','accuracy','f1'],
    ['AUC-ROC','Accuracy','F1 Score'])):
    vals = dfc[m]; order = np.argsort(vals)[::-1]
    bars = ax.bar(x[order], vals.iloc[order], 0.6, color=[colors[j%5] for j in order], edgecolor='black', lw=0.5)
    ax.set_xticks(x[order]); ax.set_xticklabels(dfc.index[order], rotation=25, ha='right', fontsize=8)
    ax.set_title(t, fontsize=11, fontweight='bold'); ax.set_ylim(0, 1.05)
    for bar, val in zip(bars, vals.iloc[order]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008, f'{val:.3f}', ha='center', va='bottom', fontsize=7)
plt.tight_layout(); plt.savefig('figures/gnn_vs_classical.png', dpi=150, bbox_inches='tight'); plt.show()

## 16. Save

In [ ]:
os.makedirs('models', exist_ok=True)
torch.save(gnn.state_dict(), 'models/SimpleGNN_binary.pth')
torch.save(cgcnn.state_dict(), 'models/CGCNN_binary.pth')
torch.save(lstm.state_dict(), 'models/LSTM_binary.pth')
pickle.dump({'SimpleGNN':res_gnn,'CGCNN':res_cgcnn,'LSTM':res_lstm,'classical':classical_res,'top3':top3_names},
            open('models/gnn_results.pkl','wb'))
print('Saved.')